# 01. Формирование DuckDB и витрин FHVHV

Читаются 16 исходных parquet-файлов за январь 2025 - апрель 2026 и
собираются витрины для аналитики и дашборда.

## 1. Настройки и источники

In [ ]:
from pathlib import Path

import duckdb
import pandas as pd


ROOT = Path.cwd()
DATA_DIR = ROOT / 'data'
ARCHIVE_DIR = DATA_DIR / 'archive'
MART_DIR = DATA_DIR / 'marts_product'
DB_PATH = DATA_DIR / 'fhvhv_product_analytics.duckdb'
PARQUET_GLOB = (ARCHIVE_DIR / 'fhvhv_tripdata_*.parquet').as_posix()

NOTEBOOK_MARTS = {
    'wav_dashboard_cells',
    'wav_dashboard_wait_summary',
    'wav_v2_date_hour',
    'wav_v2_monthly_metrics',
    'wav_v2_zone_metrics',
}
PROJECT_MARTS = NOTEBOOK_MARTS | {'grafana_wait_hist_daily'}

MART_DIR.mkdir(parents=True, exist_ok=True)
source_files = sorted(ARCHIVE_DIR.glob('fhvhv_tripdata_*.parquet'))
assert len(source_files) == 16, (
    f'Ожидалось 16 parquet-файлов, найдено {len(source_files)}'
)

## 2. Единый слой признаков

`core_valid` задаёт общий знаменатель витрин спроса, сегментов и географии.
Для ожидания, посадки и финансов используются отдельные признаки валидности.

In [ ]:
con = duckdb.connect(str(DB_PATH))
con.execute("SET memory_limit = '4GB'")
con.execute('SET threads = 4')
con.execute("SET temp_directory = 'data/duckdb_product_tmp'")
con.execute("SET default_null_order = 'NULLS LAST'")

con.execute(f'''
CREATE OR REPLACE VIEW raw_trips AS
SELECT *
FROM read_parquet('{PARQUET_GLOB}', union_by_name = true)
''')

con.execute('''
CREATE OR REPLACE VIEW trips_enriched AS
WITH base AS (
    SELECT
        *,
        CASE hvfhs_license_num
            WHEN 'HV0003' THEN 'Uber'
            WHEN 'HV0005' THEN 'Lyft'
            ELSE hvfhs_license_num
        END AS provider,
        CAST(pickup_datetime AS DATE) AS pickup_date,
        date_trunc('month', pickup_datetime)::DATE AS pickup_month,
        extract(dow FROM pickup_datetime)::INTEGER AS dow,
        extract(hour FROM pickup_datetime)::INTEGER AS pickup_hour,
        date_diff('second', request_datetime, pickup_datetime) / 60.0
            AS request_to_pickup_min,
        date_diff('second', request_datetime, on_scene_datetime) / 60.0
            AS arrival_offset_min,
        date_diff('second', on_scene_datetime, pickup_datetime) / 60.0
            AS boarding_time_min,
        date_diff('second', pickup_datetime, dropoff_datetime) / 60.0
            AS timestamp_trip_min,
        CASE WHEN trip_time > 0
             THEN trip_miles * 3600.0 / trip_time END AS speed_mph,
        COALESCE(wav_request_flag = 'Y', FALSE) AS is_wav_request,
        COALESCE(wav_match_flag = 'Y', FALSE) AS is_wav_trip,
        COALESCE(access_a_ride_flag = 'Y', FALSE) AS is_aar
    FROM raw_trips
)
SELECT
    *,
    (
        pickup_datetime IS NOT NULL
        AND dropoff_datetime IS NOT NULL
        AND dropoff_datetime >= pickup_datetime
        AND trip_time IS NOT NULL AND trip_time > 0
        AND trip_miles IS NOT NULL AND trip_miles >= 0
        AND speed_mph <= 100
        AND PULocationID IS NOT NULL AND PULocationID > 0
        AND DOLocationID IS NOT NULL AND DOLocationID > 0
    ) AS core_valid,
    (
        request_datetime IS NOT NULL
        AND pickup_datetime IS NOT NULL
        AND request_datetime <= pickup_datetime
    ) AS wait_valid,
    (
        request_datetime IS NOT NULL
        AND on_scene_datetime IS NOT NULL
        AND pickup_datetime IS NOT NULL
        AND request_datetime <= on_scene_datetime
        AND on_scene_datetime <= pickup_datetime
    ) AS arrival_wait_valid,
    (
        on_scene_datetime IS NOT NULL
        AND pickup_datetime IS NOT NULL
        AND on_scene_datetime <= pickup_datetime
    ) AS boarding_valid,
    (
        base_passenger_fare IS NOT NULL AND base_passenger_fare >= 0
        AND driver_pay IS NOT NULL AND driver_pay >= 0
        AND tips IS NOT NULL AND tips >= 0
        AND tolls IS NOT NULL AND tolls >= 0
        AND bcf IS NOT NULL AND bcf >= 0
        AND sales_tax IS NOT NULL AND sales_tax >= 0
        AND congestion_surcharge IS NOT NULL AND congestion_surcharge >= 0
        AND airport_fee IS NOT NULL AND airport_fee >= 0
        AND cbd_congestion_fee IS NOT NULL AND cbd_congestion_fee >= 0
    ) AS economics_valid
FROM base
''')

## 3. Создание витрин

Витрина сначала заменяется в DuckDB, затем экспортируется в parquet с тем же
именем.

In [ ]:
build_records = []


def build_mart(name, query, description):
    if name not in NOTEBOOK_MARTS:
        raise ValueError(f'Витрина {name} не входит в контракт проекта')

    con.execute(f'CREATE OR REPLACE TABLE {name} AS {query}')
    output_path = MART_DIR / f'{name}.parquet'
    output_path.unlink(missing_ok=True)
    con.execute(
        f"COPY {name} TO '{output_path.as_posix()}' "
        "(FORMAT PARQUET, COMPRESSION ZSTD)"
    )
    build_records.append({
        'mart': name,
        'rows': con.execute(f'SELECT COUNT(*) FROM {name}').fetchone()[0],
        'size_mb': output_path.stat().st_size / 1024 ** 2,
        'description': description,
    })


exact_wav_segment = '''
CASE
    WHEN NOT is_wav_trip AND NOT is_aar AND NOT is_wav_request
        THEN 'Не WAV, не AAR, не WAV request'
    WHEN is_wav_trip AND NOT is_aar AND NOT is_wav_request
        THEN 'WAV без AAR и WAV request'
    WHEN NOT is_wav_trip AND is_aar AND NOT is_wav_request
        THEN 'AAR без WAV и WAV request'
    WHEN NOT is_wav_trip AND NOT is_aar AND is_wav_request
        THEN 'WAV request без WAV и AAR'
    WHEN is_wav_trip AND NOT is_aar AND is_wav_request
        THEN 'WAV + WAV request, не AAR'
    WHEN NOT is_wav_trip AND is_aar AND is_wav_request
        THEN 'AAR + WAV request, не WAV'
    WHEN is_wav_trip AND is_aar AND NOT is_wav_request
        THEN 'WAV + AAR, без WAV request'
    WHEN is_wav_trip AND is_aar AND is_wav_request
        THEN 'WAV + AAR + WAV request'
END
'''

accessibility_segment = '''
CASE
    WHEN is_wav_request THEN 'WAV request'
    WHEN is_aar THEN 'AAR без WAV request'
    ELSE 'Остальные'
END
'''

### 3.1. Витрины дашборда и проверки гипотез

In [ ]:
build_mart(
    'wav_dashboard_cells',
    '''
    SELECT
        pickup_date AS date,
        dow,
        pickup_hour AS hour,
        provider,
        PULocationID,
        COUNT(*) AS all_trips,
        COUNT(*) FILTER (WHERE is_wav_trip) AS wav_trips,
        COUNT(*) FILTER (WHERE is_wav_request OR is_aar)
            AS accessibility_trips,
        COUNT(*) FILTER (WHERE NOT (is_wav_request OR is_aar))
            AS ordinary_trips,
        COUNT(*) FILTER (
            WHERE is_wav_trip AND NOT (is_wav_request OR is_aar)
        ) AS ordinary_wav_trips,
        COUNT(*) FILTER (
            WHERE NOT is_wav_trip AND NOT (is_wav_request OR is_aar)
        ) AS ordinary_nonwav_trips
    FROM trips_enriched
    WHERE core_valid
      AND provider IN ('Uber', 'Lyft')
    GROUP BY 1, 2, 3, 4, 5
    ORDER BY 1, 2, 3, 4, 5
    ''',
    'Дата, час, компания и зона для KPI, спроса и сезонности.',
)

build_mart(
    'wav_dashboard_wait_summary',
    '''
    WITH prepared AS (
        SELECT
            provider,
            CASE WHEN is_wav_request OR is_aar
                 THEN 'Инклюзивные' ELSE 'Обычные' END AS segment,
            CASE
                WHEN request_datetime IS NULL OR pickup_datetime IS NULL
                    THEN NULL
                WHEN is_wav_trip AND on_scene_datetime IS NOT NULL
                     AND arrival_wait_valid
                    THEN arrival_offset_min
                WHEN is_wav_trip AND on_scene_datetime IS NOT NULL
                    THEN NULL
                WHEN wait_valid THEN request_to_pickup_min
            END AS adjusted_wait_min,
            is_wav_trip AND on_scene_datetime IS NOT NULL
                AND arrival_wait_valid AS used_on_scene,
            (NOT is_wav_trip OR on_scene_datetime IS NULL)
                AND wait_valid AS used_pickup_proxy,
            (
                is_wav_trip AND on_scene_datetime IS NOT NULL
                AND request_datetime > on_scene_datetime
            ) OR (
                (NOT is_wav_trip OR on_scene_datetime IS NULL)
                AND request_datetime > pickup_datetime
            ) AS excluded_negative,
            is_wav_trip AND on_scene_datetime IS NOT NULL
                AND on_scene_datetime > pickup_datetime
                AS excluded_on_scene_after_pickup
        FROM trips_enriched
        WHERE core_valid
          AND provider IN ('Uber', 'Lyft')
    )
    SELECT
        CASE WHEN GROUPING(provider) = 1 THEN 'Все' ELSE provider END
            AS provider,
        segment,
        COUNT(*) AS trips,
        COUNT(adjusted_wait_min) AS valid_wait_n,
        COUNT(*) FILTER (WHERE used_on_scene) AS on_scene_wait_n,
        COUNT(*) FILTER (WHERE used_pickup_proxy) AS pickup_proxy_n,
        COUNT(*) FILTER (WHERE excluded_negative) AS excluded_negative_n,
        COUNT(*) FILTER (WHERE excluded_on_scene_after_pickup)
            AS excluded_on_scene_after_pickup_n,
        AVG(adjusted_wait_min) AS mean_min,
        approx_quantile(adjusted_wait_min, 0.50) AS p50_min,
        approx_quantile(adjusted_wait_min, 0.95) AS p95_min
    FROM prepared
    GROUP BY GROUPING SETS ((provider, segment), (segment))
    ORDER BY GROUPING(provider), provider, segment
    ''',
    'Покрытие, среднее, медиана и p95 скорректированного ожидания.',
)

### 3.2. Витрины WAV/AAR-анализа

In [ ]:
build_mart(
    'wav_v2_date_hour',
    f'''
    SELECT
        pickup_date AS date,
        pickup_hour AS hour,
        provider,
        {exact_wav_segment} AS exact_segment,
        {accessibility_segment} AS accessibility_segment,
        COUNT(*) AS trips
    FROM trips_enriched
    WHERE core_valid
      AND provider IN ('Uber', 'Lyft')
    GROUP BY 1, 2, 3, 4, 5
    ORDER BY 1, 2, 3, 4, 5
    ''',
    'Дата, час и компания для комбинаций WAV, AAR и WAV request.',
)

build_mart(
    'wav_v2_monthly_metrics',
    f'''
    WITH prepared AS (
        SELECT
            pickup_month AS month,
            provider,
            {accessibility_segment} AS accessibility_segment,
            * EXCLUDE (pickup_month, provider)
        FROM trips_enriched
        WHERE core_valid
          AND provider IN ('Uber', 'Lyft')
    )
    SELECT
        CASE WHEN GROUPING(month) = 1 THEN NULL ELSE month END AS month,
        CASE WHEN GROUPING(provider) = 1 THEN 'Все' ELSE provider END
            AS provider,
        accessibility_segment,
        COUNT(*) AS trips,
        COUNT(*) FILTER (WHERE is_wav_request) AS wav_request_n,
        COUNT(*) FILTER (WHERE is_wav_request AND is_wav_trip) AS wav_match_n,
        COUNT(*) FILTER (WHERE arrival_wait_valid) AS arrival_n,
        AVG(arrival_offset_min) FILTER (WHERE arrival_wait_valid)
            AS arrival_mean_min,
        approx_quantile(arrival_offset_min, 0.50)
            FILTER (WHERE arrival_wait_valid) AS arrival_p50_min,
        approx_quantile(arrival_offset_min, 0.95)
            FILTER (WHERE arrival_wait_valid) AS arrival_p95_min,
        COUNT(*) FILTER (WHERE boarding_valid) AS boarding_n,
        AVG(boarding_time_min) FILTER (WHERE boarding_valid)
            AS boarding_mean_min,
        approx_quantile(boarding_time_min, 0.50)
            FILTER (WHERE boarding_valid) AS boarding_p50_min,
        approx_quantile(boarding_time_min, 0.95)
            FILTER (WHERE boarding_valid) AS boarding_p95_min,
        COUNT(*) FILTER (WHERE economics_valid AND trip_miles > 0)
            AS economics_n,
        SUM(base_passenger_fare)
            FILTER (WHERE economics_valid AND trip_miles > 0)
            / NULLIF(
                SUM(trip_miles)
                    FILTER (WHERE economics_valid AND trip_miles > 0), 0
            ) AS base_fare_per_mile,
        SUM(base_passenger_fare)
            FILTER (WHERE economics_valid AND timestamp_trip_min > 0)
            / NULLIF(
                SUM(timestamp_trip_min)
                    FILTER (WHERE economics_valid AND timestamp_trip_min > 0), 0
            ) AS base_fare_per_min,
        AVG(trip_miles) FILTER (WHERE economics_valid AND trip_miles >= 0)
            AS mean_miles,
        AVG(timestamp_trip_min)
            FILTER (WHERE economics_valid AND timestamp_trip_min > 0)
            AS mean_duration_min,
        60.0 * SUM(trip_miles)
            FILTER (WHERE economics_valid AND timestamp_trip_min > 0)
            / NULLIF(
                SUM(timestamp_trip_min)
                    FILTER (WHERE economics_valid AND timestamp_trip_min > 0), 0
            ) AS aggregate_speed_mph,
        COUNT(*) FILTER (WHERE economics_valid AND tips > 0) AS tipped_n,
        100.0 * AVG((tips > 0)::INTEGER) FILTER (WHERE economics_valid)
            AS tipped_share_pct,
        AVG(tips) FILTER (WHERE economics_valid AND tips > 0)
            AS mean_tip_positive
    FROM prepared
    GROUP BY GROUPING SETS (
        (month, provider, accessibility_segment),
        (month, accessibility_segment),
        (provider, accessibility_segment),
        (accessibility_segment)
    )
    ORDER BY month NULLS LAST, provider, accessibility_segment
    ''',
    'Помесячные метрики спроса, ожидания, поездки, стоимости и чаевых.',
)

build_mart(
    'wav_v2_zone_metrics',
    '''
    WITH pickup AS (
        SELECT
            'Pickup' AS direction,
            PULocationID AS LocationID,
            COUNT(*) AS total_trips,
            COUNT(*) FILTER (WHERE is_wav_request OR is_aar)
                AS accessibility_trips,
            COUNT(*) FILTER (WHERE is_wav_request) AS wav_request_trips,
            COUNT(*) FILTER (WHERE is_aar AND NOT is_wav_request)
                AS aar_no_request_trips
        FROM trips_enriched
        WHERE core_valid
          AND provider IN ('Uber', 'Lyft')
        GROUP BY 2
    ), dropoff AS (
        SELECT
            'Dropoff' AS direction,
            DOLocationID AS LocationID,
            COUNT(*) AS total_trips,
            COUNT(*) FILTER (WHERE is_wav_request OR is_aar)
                AS accessibility_trips,
            COUNT(*) FILTER (WHERE is_wav_request) AS wav_request_trips,
            COUNT(*) FILTER (WHERE is_aar AND NOT is_wav_request)
                AS aar_no_request_trips
        FROM trips_enriched
        WHERE core_valid
          AND provider IN ('Uber', 'Lyft')
        GROUP BY 2
    ), combined AS (
        SELECT * FROM pickup
        UNION ALL
        SELECT * FROM dropoff
    )
    SELECT
        *,
        100.0 * accessibility_trips / total_trips
            AS accessibility_share_pct
    FROM combined
    ORDER BY 1, 2
    ''',
    'Зона pickup/dropoff, объём и доля инклюзивных поездок.',
)